# Approach events and neural responses

End-to-end first pass: classical tracking (`mouse_tracker.py`) -> kinematics on the ephys clock (`build_events.py`, `approach_events.py`) -> approach episodes per wall -> spike responses of good units around investigation onset.

**Tracking caveats (read before interpreting).** The pretrained DeepLabCut SuperAnimal model did not transfer to this IR footage (detections in 3% of frames), so positions come from background subtraction on the dark trunk of the mouse. Body position is reliable; head direction is inferred mostly from direction of travel and is unreliable when the mouse is stationary or turning in place. Events that depend on head angle (target commitment) are therefore provisional. The arena is a 1.5 ft (45.7 cm) glass box.

**Wall interfaces are empirical.** Each wall's interface is the polyline where the nose actually reaches (0.5th percentile per bin), because the camera perspective and an obstruction along the top wall make the rim polygon unusable. Zones are 3 cm from that line.

**Stimuli.** The three cages sit along the **left** wall (camera coordinates): male at the top, fake mouse in the middle, female at the bottom. Each cage is taken as one third of the left wall's empirical interface. The other three walls (top / right / bottom) are non-social controls. Note that the left wall received the fewest visits and the top wall the most, so the wall assignment is worth double-checking against the video orientation.

In [ ]:
%matplotlib inline
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import npx_io as io, approach_events as ae
from arena import Arena, to_small, RIM_ORIG
sns.set_context('notebook'); sns.set_style('ticks')
DATA_DIR = Path('../../preliminary data /Neuropixel_open_field').resolve(); CACHE = Path('cache')
FPS = 59.63700234192037
import pickle
arena = pickle.load(open(CACHE / 'arena_session.pkl', 'rb'))          # empirical interfaces + cage targets, built by build_events.py
SOCIAL = ['male', 'fake', 'female']; CONTROL = ['top', 'right', 'bottom']
print('targets:', list(arena.walls))
trk = pd.read_parquet(CACHE / 'tracking_session.parquet'); kin = pd.read_parquet(CACHE / 'kinematics_session.parquet')
ev = pd.read_csv(CACHE / 'approach_events_session.csv')
print(f'{len(trk)} frames, {trk.t_ephys.min():.0f}-{trk.t_ephys.max():.0f} s ephys, tracked {trk.mouse_center_x.notna().mean()*100:.1f}%, '
      f'directly detected {(trk.area > 0).mean()*100:.1f}%, held {(trk.area == 0).mean()*100:.1f}%')
print(f'{len(ev)} episodes; px/cm = {arena.px_per_cm:.2f} (45.7 cm arena)')

## 1. Tracking quality and occupancy

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
ax = axes[0]
ax.hist(kin.speed.dropna(), bins=np.arange(0, 60, 1), color='k'); ax.set_xlabel('body speed (cm/s)'); ax.set_ylabel('frames'); ax.set_yscale('log')
ax.set_title(f'speed; median {kin.speed.median():.1f}, frac > 4 cm/s = {np.nanmean(kin.speed > 4):.2f}')
ax = axes[1]
h, xe, ye = np.histogram2d(kin.body_x.dropna(), kin.body_y.dropna(), bins=60)
ax.imshow(np.log1p(h.T), origin='upper', cmap='magma', extent=[xe[0], xe[-1], ye[-1], ye[0]])
ax.plot(*np.vstack([arena.rim, arena.rim[:1]]).T, 'c-', lw=1)
for w, d in arena.walls.items():
    ax.plot(d['poly'][:, 0], d['poly'][:, 1], 'y-', lw=3); ax.text(*(d['poly'].mean(0) + d['inward'] * 25), w, color='y', ha='center')
ax.set_title('occupancy (log), empirical wall interfaces in yellow'); ax.set_xticks([]); ax.set_yticks([])
ax = axes[2]
zone = pd.DataFrame({w: kin[f'd_{w}'] < ae.DEFAULTS['d_zone_cm'] for w in arena.walls})
zone['t'] = kin.t_ephys
occ = zone.groupby(zone.t // 300).mean(numeric_only=True) * 100
occ.index = occ.index * 5
occ[list(arena.walls)].plot(ax=ax, marker='o'); ax.set_xlabel('session time (min)'); ax.set_ylabel('% time snout within 3 cm of segment'); ax.set_title('wall-zone occupancy over the session')
sns.despine(); plt.tight_layout()
print('total % time in each wall zone:'); print((zone[list(arena.walls)].mean() * 100).round(2).to_string())

## 2. Approach episodes
Outcome categories follow the planning document. `investigate_no_commit` means the snout reached a wall segment without a sustained head-aligned approach being detected first, which with the current heading estimate is expected to be common.

In [ ]:
print(ev.outcome.value_counts().to_string()); print()
print(pd.crosstab(ev.invest_wall.fillna('none'), ev.outcome))
inv = ev[ev.invest.notna()].copy()
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
inv.invest_wall.value_counts().reindex(list(arena.walls)).plot.bar(ax=axes[0], color='k'); axes[0].set_ylabel('n investigations'); axes[0].set_title('investigations per wall')
axes[1].hist(inv.invest_t / 60, bins=np.arange(0, 96, 5), color='k'); axes[1].set_xlabel('session time (min)'); axes[1].set_ylabel('n investigations'); axes[1].set_title('timing')
dur = (inv.withdraw - inv.invest) / FPS
axes[2].hist(dur.dropna(), bins=np.arange(0, 20, 0.5), color='k'); axes[2].set_xlabel('investigation to withdrawal (s)'); axes[2].set_title('investigation duration')
sns.despine(); plt.tight_layout()

In [ ]:
# example trajectories for the first 12 successful or investigate episodes
sel = inv.head(12)
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, (_, e) in zip(axes.ravel(), sel.iterrows()):
    a, b = int(e.onset) - 60, int(e.invest if np.isnan(e.withdraw) else e.withdraw) + 60
    seg = kin.iloc[max(a, 0):b]
    ax.plot(*np.vstack([arena.rim, arena.rim[:1]]).T, 'k-', lw=1)
    sc = ax.scatter(seg.nose_x, seg.nose_y, c=(seg.frame - e.onset) / FPS, cmap='viridis', s=4)
    ax.plot(seg.body_x, seg.body_y, color='grey', lw=.5)
    ax.set_title(f"{e.invest_wall}: {e.outcome} @ {e.invest_t/60:.1f} min", fontsize=9); ax.set_xticks([]); ax.set_yticks([]); ax.invert_yaxis()
plt.tight_layout()

## 3. Neural responses around investigation onset
Good units, aligned to the first frame the snout enters a wall's interaction zone. Population mean per wall first, then per-unit responses. This is descriptive; the planning document's GLM with behavioral regressors comes next once heading is trustworthy.

In [ ]:
ci = io.load_cluster_info(DATA_DIR); st, sc = io.load_spikes(DATA_DIR)
good = ci[ci.label == 'good'].sort_values('depth_um'); units = good.index.to_numpy()
spk = io.spikes_by_unit(st, sc, units)
W_PRE, W_POST, BIN = 3.0, 3.0, 0.1
edges = np.arange(-W_PRE, W_POST + BIN, BIN); tb = edges[:-1] + BIN / 2

def psth(unit_ids, t_events):
    out = np.zeros((len(unit_ids), len(t_events), len(tb)))
    for i, u in enumerate(unit_ids):
        t = spk[u]
        for j, te in enumerate(t_events):
            seg = t[(t >= te - W_PRE) & (t < te + W_POST)] - te
            out[i, j] = np.histogram(seg, edges)[0] / BIN
    return out

walls = list(arena.walls); resp = {}
for w in walls:
    te = inv.loc[inv.invest_wall == w, 'invest_t'].dropna().values
    te = te[(te > W_PRE) & (te < st.max() - W_POST)]
    if len(te) >= 8: resp[w] = psth(units, te)          # targets with fewer than 8 events are excluded
print({w: r.shape[1] for w, r in resp.items()}, 'events per wall')
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for w, r in resp.items():
    m = r.mean(1)                                  # units x time
    base = m[:, tb < -1].mean(1, keepdims=True)
    z = (m - base) / (m[:, tb < -1].std(1, keepdims=True) + 0.5)
    axes[0].plot(tb, m.mean(0), label=f'{w} (n={r.shape[1]})'); axes[1].plot(tb, z.mean(0), label=w)
axes[0].set_ylabel('mean rate, good units (Hz)'); axes[1].set_ylabel('mean z vs pre-window')
for ax in axes: ax.axvline(0, color='k', lw=.8); ax.set_xlabel('time from investigation onset (s)'); ax.legend(frameon=False)
sns.despine(); plt.tight_layout()

In [ ]:
# per-unit heatmaps sorted by response, one panel per wall, plus a random-time control sorted the same way
rng = np.random.default_rng(0)
n_ctrl = int(np.median([r.shape[1] for r in resp.values()]))
t_rand = rng.uniform(W_PRE + 10, st.max() - W_POST - 10, n_ctrl)
resp_ctrl = psth(units, t_rand)
panels = list(resp.items()) + [('random times', resp_ctrl)]
fig, axes = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 7), sharey=True)
for ax, (w, r) in zip(axes, panels):
    m = r.mean(1); base = m[:, tb < -1]
    z = (m - base.mean(1, keepdims=True)) / (base.std(1, keepdims=True) + 0.5)
    order = np.argsort(-z[:, (tb > 0) & (tb < 1)].mean(1))
    im = ax.imshow(z[order], aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2, extent=[tb[0], tb[-1], len(units), 0])
    ax.axvline(0, color='k', lw=.8); ax.set_title(f'{w} (n={r.shape[1]})'); ax.set_xlabel('s from investigation')
axes[0].set_ylabel('good units (sorted by 0-1 s response)'); plt.colorbar(im, ax=axes, fraction=.02, label='z')
# how many units exceed |z| > 1 in the 0-1 s window, real vs random alignment
def n_resp(r):
    m = r.mean(1); base = m[:, tb < -1]; z = (m - base.mean(1, keepdims=True)) / (base.std(1, keepdims=True) + 0.5)
    v = z[:, (tb > 0) & (tb < 1)].mean(1); return int((v > 1).sum()), int((v < -1).sum())
for w, r in panels: print(f'{w:13s} n={r.shape[1]:3d}: units with mean z > 1 in 0-1 s: {n_resp(r)[0]:3d}, z < -1: {n_resp(r)[1]:3d}')

### Wall selectivity (descriptive)
For each unit, spike counts in the 0 to 1 s window after investigation onset are compared across walls (walls with at least 10 events) with a Kruskal-Wallis test, and the same test is run on random-time alignments split into pseudo-walls with matched counts as a false-positive check. With unequal event counts and no behavioral covariates this is only a screen; the GLM in the planning document is the real test.

In [ ]:
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests
win = (tb > 0) & (tb < 1); pre = (tb > -3) & (tb < -1)
# baseline-subtracted response per event (0-1 s minus -3..-1 s), so slow rate differences by location do not count
counts = {w: r[:, :, win].mean(2) - r[:, :, pre].mean(2) for w, r in resp.items()}     # units x events
p = np.array([kruskal(*[counts[w][i] for w in counts]).pvalue for i in range(len(units))])
rej = multipletests(p, method='fdr_bh')[0]
print(f'{rej.sum()} of {len(units)} good units differ across walls at FDR 0.05 (baseline-subtracted 0-1 s response)')
ns = [r.shape[1] for r in resp.values()]
t_rand = rng.uniform(W_PRE + 10, st.max() - W_POST - 10, sum(ns)); rc_all = psth(units, t_rand)
rc = rc_all[:, :, win].mean(2) - rc_all[:, :, pre].mean(2)
splits = np.split(np.arange(sum(ns)), np.cumsum(ns)[:-1])
p0 = np.array([kruskal(*[rc[i, s] for s in splits]).pvalue for i in range(len(units))])
print(f'random-time control: {multipletests(p0, method="fdr_bh")[0].sum()} of {len(units)} units "selective" at FDR 0.05')
# which wall drives it: mean baseline-subtracted response per wall for the selective units
resp_tab = pd.DataFrame({w: counts[w].mean(1) for w in counts}, index=units)
print('mean baseline-subtracted response (Hz) of selective units, per wall:'); print(resp_tab[rej].mean().round(2).to_string())
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
order_sel = np.argsort(p)[:8]
for ax, i in zip(axes.ravel(), order_sel):
    for w, r in resp.items(): ax.plot(tb, r[i].mean(0), label=w)
    ax.axvline(0, color='k', lw=.8); ax.set_title(f'unit {units[i]}  {good.depth_um.iloc[i]:.0f} µm  p={p[i]:.1e}', fontsize=10)
axes[0, 0].legend(frameon=False, fontsize=8); axes[1, 0].set_xlabel('s from investigation'); axes[0, 0].set_ylabel('Hz')
sns.despine(); plt.tight_layout()

### Social cages vs non-social walls
Pool investigations of the three cages and compare with the three control walls, then compare the cages with each other. Baseline-subtracted 0 to 1 s responses, Mann-Whitney / Kruskal-Wallis per unit with FDR, random-time alignments as the false-positive check.

In [ ]:
from scipy.stats import mannwhitneyu
def resp_events(names):
    return {w: counts[w] for w in names if w in counts}
soc, ctl = resp_events(SOCIAL), resp_events(CONTROL)
if soc and ctl:
    S = np.concatenate(list(soc.values()), 1); C = np.concatenate(list(ctl.values()), 1)
    p_sc = np.array([mannwhitneyu(S[i], C[i]).pvalue for i in range(len(units))])
    rej_sc = multipletests(p_sc, method='fdr_bh')[0]
    print(f'social ({S.shape[1]} events) vs control walls ({C.shape[1]} events): {rej_sc.sum()} of {len(units)} units differ at FDR 0.05; '
          f'{(rej_sc & (S.mean(1) > C.mean(1))).sum()} higher for social, {(rej_sc & (S.mean(1) < C.mean(1))).sum()} lower')
    # control: random split of the pooled events with the same sizes
    pooled = np.concatenate([S, C], 1); idx = rng.permutation(pooled.shape[1]); Sr, Cr = pooled[:, idx[:S.shape[1]]], pooled[:, idx[S.shape[1]:]]
    p_r = np.array([mannwhitneyu(Sr[i], Cr[i]).pvalue for i in range(len(units))])
    print(f'label-shuffle control: {multipletests(p_r, method="fdr_bh")[0].sum()} units')
if len(soc) >= 2:
    p_3 = np.array([kruskal(*[soc[w][i] for w in soc]).pvalue for i in range(len(units))])
    rej_3 = multipletests(p_3, method='fdr_bh')[0]
    print(f'between cages ({ {w: v.shape[1] for w, v in soc.items()} }): {rej_3.sum()} units differ at FDR 0.05')
    tab = pd.DataFrame({w: soc[w].mean(1) for w in soc}, index=units)
    print('mean baseline-subtracted response (Hz) of cage-selective units:'); print(tab[rej_3].mean().round(2).to_string())
    fig, axes = plt.subplots(2, 4, figsize=(16, 7))
    for ax, i in zip(axes.ravel(), np.argsort(p_3)[:8]):
        for w in SOCIAL + CONTROL:
            if w in resp: ax.plot(tb, resp[w][i].mean(0), label=w, lw=1.5 if w in SOCIAL else 0.8, alpha=1 if w in SOCIAL else 0.5)
        ax.axvline(0, color='k', lw=.8); ax.set_title(f'unit {units[i]}  {good.depth_um.iloc[i]:.0f} µm  p={p_3[i]:.1e}', fontsize=10)
    axes[0, 0].legend(frameon=False, fontsize=8); axes[1, 0].set_xlabel('s from investigation'); axes[0, 0].set_ylabel('Hz')
    sns.despine(); plt.tight_layout()

## 4. Reading these results

* **The top wall is different.** It receives the most investigations, the mouse spends the most zone time there, and that share climbs late in the session. If the stimulus bank sat along the top wall this is the expected pattern, but it could also be a resting corner. The session notes decide this.
* **Investigation onset is a real neural event.** Around 70 units rise and 50 fall by more than one z after arrival at a wall, against about 10 each for random alignments. Much of this is approach locomotion and its end: the population rate rises about 1 s before onset and falls after.
* **Wall differences are confounded with position and movement.** A unit that fires more near the top wall will look "selective" whether or not a stimulus is there. Baseline subtraction removes slow location effects but not approach kinematics. The planning document's GLM with speed, distance and session-time regressors is the proper test.

## 5. Next steps
* Assign stimulus identities to walls from the session notes, then re-run with counterbalancing information if any.
* Head direction: label ~200 frames and fine-tune SuperAnimal, or add a second camera. Commitment-locked analyses wait on this.
* Fit the continuous-time GLM (speed, wall distance, session time, dip state, plus stimulus x peri-event basis) per the planning document.